In [4]:
# Step 1.1：定位数据并确认文件数量
from pathlib import Path
from time import perf_counter

# r 表示按原样读取 Windows 路径
DATA_ROOT = Path(r"<HOME_CREDIT_DATA_ROOT>")
PROJECT_ROOT = Path(r"<PROJECT_ROOT>")
METADATA_DIR = PROJECT_ROOT / "metadata"

METADATA_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"数据目录不存在：{DATA_ROOT}")

start = perf_counter()

# 这里只扫描文件名，不读取 Parquet 数据内容
parquet_files = sorted(DATA_ROOT.glob("**/*.parquet"))
feature_definition_files = sorted(
    DATA_ROOT.glob("**/feature_definitions.csv")
)

elapsed = perf_counter() - start

print("数据集目录：", DATA_ROOT.resolve())
print("项目目录：", PROJECT_ROOT.resolve())
print("metadata 输出目录：", METADATA_DIR.resolve())
print("发现的 Parquet 文件数：", len(parquet_files))
print("feature_definitions.csv 数量：", len(feature_definition_files))
print(f"扫描耗时：{elapsed:.2f} 秒")

print("\n前 10 个 Parquet 文件：")
for path in parquet_files[:10]:
    print(" -", path.relative_to(DATA_ROOT))

print("\nfeature_definitions.csv 路径：")
for path in feature_definition_files:
    print(" -", path.relative_to(DATA_ROOT))

数据集目录： <HOME_CREDIT_DATA_ROOT>
项目目录： <PROJECT_ROOT>
metadata 输出目录： <PROJECT_ROOT>\metadata
发现的 Parquet 文件数： 68
feature_definitions.csv 数量： 1
扫描耗时：0.00 秒

前 10 个 Parquet 文件：
 - parquet_files\test\test_applprev_1_0.parquet
 - parquet_files\test\test_applprev_1_1.parquet
 - parquet_files\test\test_applprev_1_2.parquet
 - parquet_files\test\test_applprev_2.parquet
 - parquet_files\test\test_base.parquet
 - parquet_files\test\test_credit_bureau_a_1_0.parquet
 - parquet_files\test\test_credit_bureau_a_1_1.parquet
 - parquet_files\test\test_credit_bureau_a_1_2.parquet
 - parquet_files\test\test_credit_bureau_a_1_3.parquet
 - parquet_files\test\test_credit_bureau_a_1_4.parquet

feature_definitions.csv 路径：
 - feature_definitions.csv


In [5]:
# Step 1.2：读取字段定义
import pandas as pd

FEATURE_DEFINITIONS_PATH = feature_definition_files[0]

feature_definitions = pd.read_csv(FEATURE_DEFINITIONS_PATH)

print("文件路径：", FEATURE_DEFINITIONS_PATH)
print("表格形状（行数, 列数）：", feature_definitions.shape)
print("\n字段名称：")
print(feature_definitions.columns.tolist())

print("\n前 10 行：")
display(feature_definitions.head(10))

print("\n各列缺失值数量：")
display(feature_definitions.isna().sum().to_frame("missing_count"))

print("\n各列数据类型：")
display(feature_definitions.dtypes.to_frame("dtype"))

文件路径： <HOME_CREDIT_DATA_ROOT>\feature_definitions.csv
表格形状（行数, 列数）： (465, 2)

字段名称：
['Variable', 'Description']

前 10 行：


,Variable,Description
0,actualdpd_943P,Days Past Due (DPD) of previous contract (actu...
1,actualdpdtolerance_344P,DPD of client with tolerance.
2,addres_district_368M,District of the person's address.
3,addres_role_871L,Role of person's address.
4,addres_zip_823M,Zip code of the address.
5,amount_1115A,Credit amount of the active contract provided ...
6,amount_416A,Deposit amount.
7,amount_4527230A,Tax deductions amount tracked by the governmen...
8,amount_4917619A,Tax deductions amount tracked by the governmen...
9,amtdebitincoming_4809443A,Incoming debit card transactions amount.



各列缺失值数量：


,missing_count
Variable,0
Description,0



各列数据类型：


,dtype
Variable,object
Description,object


In [6]:
#Step 1.3：从文件名识别数据集、表组、depth 和分片
import re
import pandas as pd


def parse_parquet_name(path):
    """
    例如：
    test_applprev_1_0.parquet
    -> split=test
    -> table_group=applprev
    -> depth=1
    -> shard=0
    """
    stem = path.stem

    split_match = re.match(r"^(train|test)_(.+)$", stem)

    if split_match is None:
        split = "unknown"
        body = stem
    else:
        split = split_match.group(1)
        body = split_match.group(2)

    # base 是特殊的 case-level 锚点表
    if body == "base":
        table_group = "base"
        depth = None
        shard = None
        table_role = "base"

    # 例如 credit_bureau_a_1_3、static_0_0
    elif match := re.match(r"^(.*)_([012])_(\d+)$", body):
        table_group = match.group(1)
        depth = int(match.group(2))
        shard = int(match.group(3))
        table_role = "feature_table"

    # 例如 applprev_2、person_1
    elif match := re.match(r"^(.*)_([012])$", body):
        table_group = match.group(1)
        depth = int(match.group(2))
        shard = None
        table_role = "feature_table"

    else:
        table_group = body
        depth = None
        shard = None
        table_role = "unparsed"

    return {
        "split": split,
        "table_group": table_group,
        "depth": depth,
        "shard": shard,
        "table_role": table_role
    }


records = []

for path in parquet_files:
    parsed = parse_parquet_name(path)

    records.append({
        "file_name": path.name,
        "relative_path": str(path.relative_to(DATA_ROOT)),
        "size_bytes": path.stat().st_size,
        "size_mb": round(path.stat().st_size / 1024**2, 2),
        **parsed
    })

inventory = pd.DataFrame(records)

# 使用可空整数，避免 depth/shard 因缺失值变成小数
inventory["depth"] = inventory["depth"].astype("Int64")
inventory["shard"] = inventory["shard"].astype("Int64")

inventory["is_sharded"] = inventory["shard"].notna()

# 忽略 train/test 后的逻辑表结构
inventory["table_family_id"] = (
    inventory["table_group"]
    + "__depth_"
    + inventory["depth"].astype("string").fillna("base")
)

print("物理 Parquet 文件数：", len(inventory))
print("逻辑表结构数：", inventory["table_family_id"].nunique())
print("无法解析的文件数：", (inventory["table_role"] == "unparsed").sum())
print(
    "Parquet 总大小：",
    round(inventory["size_bytes"].sum() / 1024**3, 2),
    "GB"
)

print("\n按 train/test 统计物理文件：")
display(
    inventory.groupby("split")
    .size()
    .rename("file_count")
    .to_frame()
)

print("\n按 split 和 depth 统计：")
display(
    inventory.groupby(
        ["split", "depth"],
        dropna=False
    )
    .size()
    .rename("file_count")
    .reset_index()
)

print("\n识别出的逻辑表结构：")
display(
    inventory[
        ["table_group", "depth", "table_role", "table_family_id"]
    ]
    .drop_duplicates()
    .sort_values(
        ["table_role", "table_group", "depth"],
        na_position="first"
    )
    .reset_index(drop=True)
)

print("\nInventory 前 15 行：")
display(inventory.head(15))

物理 Parquet 文件数： 68
逻辑表结构数： 17
无法解析的文件数： 0
Parquet 总大小： 1.24 GB

按 train/test 统计物理文件：


,file_count
split,
test,36
train,32



按 split 和 depth 统计：


,split,depth,file_count
0,test,0,4
1,test,1,16
2,test,2,15
3,test,<NA>,1
4,train,0,3
5,train,1,14
6,train,2,14
7,train,<NA>,1



识别出的逻辑表结构：


,table_group,depth,table_role,table_family_id
0,base,<NA>,base,base__depth_base
1,applprev,1,feature_table,applprev__depth_1
2,applprev,2,feature_table,applprev__depth_2
3,credit_bureau_a,1,feature_table,credit_bureau_a__depth_1
4,credit_bureau_a,2,feature_table,credit_bureau_a__depth_2
5,credit_bureau_b,1,feature_table,credit_bureau_b__depth_1
6,credit_bureau_b,2,feature_table,credit_bureau_b__depth_2
7,debitcard,1,feature_table,debitcard__depth_1
8,deposit,1,feature_table,deposit__depth_1
9,other,1,feature_table,other__depth_1



Inventory 前 15 行：


,file_name,relative_path,size_bytes,size_mb,split,table_group,depth,shard,table_role,is_sharded,table_family_id
0,test_applprev_1_0.parquet,parquet_files\test\test_applprev_1_0.parquet,29733,0.03,test,applprev,1,0,feature_table,True,applprev__depth_1
1,test_applprev_1_1.parquet,parquet_files\test\test_applprev_1_1.parquet,30309,0.03,test,applprev,1,1,feature_table,True,applprev__depth_1
2,test_applprev_1_2.parquet,parquet_files\test\test_applprev_1_2.parquet,30289,0.03,test,applprev,1,2,feature_table,True,applprev__depth_1
3,test_applprev_2.parquet,parquet_files\test\test_applprev_2.parquet,4687,0.00,test,applprev,2,<NA>,feature_table,False,applprev__depth_2
4,test_base.parquet,parquet_files\test\test_base.parquet,3343,0.00,test,base,<NA>,<NA>,base,False,base__depth_base
5,test_credit_bureau_a_1_0.parquet,parquet_files\test\test_credit_bureau_a_1_0.pa...,59968,0.06,test,credit_bureau_a,1,0,feature_table,True,credit_bureau_a__depth_1
6,test_credit_bureau_a_1_1.parquet,parquet_files\test\test_credit_bureau_a_1_1.pa...,60903,0.06,test,credit_bureau_a,1,1,feature_table,True,credit_bureau_a__depth_1
7,test_credit_bureau_a_1_2.parquet,parquet_files\test\test_credit_bureau_a_1_2.pa...,60300,0.06,test,credit_bureau_a,1,2,feature_table,True,credit_bureau_a__depth_1
8,test_credit_bureau_a_1_3.parquet,parquet_files\test\test_credit_bureau_a_1_3.pa...,61325,0.06,test,credit_bureau_a,1,3,feature_table,True,credit_bureau_a__depth_1
9,test_credit_bureau_a_1_4.parquet,parquet_files\test\test_credit_bureau_a_1_4.pa...,60115,0.06,test,credit_bureau_a,1,4,feature_table,True,credit_bureau_a__depth_1


In [8]:
#Step 1.4：读取 Parquet 元数据
import json
import hashlib
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd


# 先检查文件名解析结果
unparsed_count = (inventory["table_role"] == "unparsed").sum()

print("物理 Parquet 文件数：", len(inventory))
print("逻辑表结构数：", inventory["table_family_id"].nunique())
print("无法解析的文件数：", unparsed_count)

if unparsed_count > 0:
    display(
        inventory.loc[
            inventory["table_role"] == "unparsed",
            ["file_name", "relative_path"]
        ]
    )


metadata_records = []
failed_files = []

for index, path in enumerate(parquet_files, start=1):
    try:
        parquet_file = pq.ParquetFile(path)
        parquet_metadata = parquet_file.metadata
        schema = parquet_file.schema_arrow

        column_names = schema.names
        schema_types = {
            field.name: str(field.type)
            for field in schema
        }

        # 真正使用 Arrow 日期或时间类型的字段
        typed_time_columns = [
            field.name
            for field in schema
            if (
                pa.types.is_date(field.type)
                or pa.types.is_timestamp(field.type)
                or pa.types.is_time(field.type)
                or pa.types.is_duration(field.type)
            )
        ]

        # base 表中还存在数值型时间索引
        special_time_columns = [
            column
            for column in ["date_decision", "WEEK_NUM", "MONTH"]
            if column in column_names
        ]

        time_columns = sorted(
            set(typed_time_columns + special_time_columns)
        )

        # 这里只是“候选连接键”，还没有验证唯一性
        key_columns_present = [
            column
            for column in ["case_id", "num_group1", "num_group2"]
            if column in column_names
        ]

        schema_json = json.dumps(
            schema_types,
            ensure_ascii=False,
            sort_keys=True
        )

        schema_fingerprint = hashlib.sha256(
            schema_json.encode("utf-8")
        ).hexdigest()[:16]

        metadata_records.append({
            "relative_path": str(path.relative_to(DATA_ROOT)),
            "row_count": parquet_metadata.num_rows,
            "row_group_count": parquet_metadata.num_row_groups,
            "column_count": len(column_names),
            "has_case_id": "case_id" in column_names,
            "has_target": "target" in column_names,
            "key_columns_present": "|".join(key_columns_present),
            "time_columns": "|".join(time_columns),
            "schema_fingerprint": schema_fingerprint,
            "schema_json": schema_json
        })

    except Exception as error:
        failed_files.append({
            "relative_path": str(path.relative_to(DATA_ROOT)),
            "error": repr(error)
        })

    if index % 10 == 0 or index == len(parquet_files):
        print(f"已读取元数据：{index}/{len(parquet_files)}")


metadata_df = pd.DataFrame(metadata_records)

# 保证重复运行代码时不会产生 _x、_y 后缀
metadata_columns = [
    "row_count",
    "row_group_count",
    "column_count",
    "has_case_id",
    "has_target",
    "key_columns_present",
    "time_columns",
    "schema_fingerprint",
    "schema_json"
]

inventory = inventory.drop(
    columns=metadata_columns,
    errors="ignore"
)

inventory = inventory.merge(
    metadata_df,
    on="relative_path",
    how="left",
    validate="one_to_one"
)


print("\n读取失败的文件数：", len(failed_files))

if failed_files:
    display(pd.DataFrame(failed_files))


print("\n按 train/test 统计：")
display(
    inventory.groupby("split").agg(
        physical_file_count=("file_name", "count"),
        table_structure_count=("table_family_id", "nunique"),
        total_physical_rows=("row_count", "sum"),
        total_size_gb=("size_bytes", lambda x: round(x.sum() / 1024**3, 2))
    ).reset_index()
)


print("\n各表结构的文件数和 Schema 数：")
table_schema_summary = (
    inventory.groupby(
        ["split", "table_group", "depth"],
        dropna=False
    )
    .agg(
        physical_file_count=("file_name", "count"),
        total_rows=("row_count", "sum"),
        column_count_min=("column_count", "min"),
        column_count_max=("column_count", "max"),
        schema_version_count=("schema_fingerprint", "nunique")
    )
    .reset_index()
)

display(table_schema_summary)


print("\n体积最大的 10 个文件：")
display(
    inventory[
        [
            "file_name",
            "split",
            "table_group",
            "depth",
            "shard",
            "size_mb",
            "row_count",
            "column_count"
        ]
    ]
    .sort_values("size_mb", ascending=False)
    .head(10)
)

物理 Parquet 文件数： 68
逻辑表结构数： 17
无法解析的文件数： 0
已读取元数据：10/68
已读取元数据：20/68
已读取元数据：30/68
已读取元数据：40/68
已读取元数据：50/68
已读取元数据：60/68
已读取元数据：68/68

读取失败的文件数： 0

按 train/test 统计：


,split,physical_file_count,table_structure_count,total_physical_rows,total_size_gb
0,test,36,17,350,0.00
1,train,32,17,243465196,1.24



各表结构的文件数和 Schema 数：


,split,table_group,depth,physical_file_count,total_rows,column_count_min,column_count_max,schema_version_count
0,test,applprev,1,3,30,41,41,2
1,test,applprev,2,1,10,6,6,1
2,test,base,<NA>,1,10,4,4,1
3,test,credit_bureau_a,1,5,50,79,79,4
4,test,credit_bureau_a,2,12,120,19,19,1
5,test,credit_bureau_b,1,1,10,45,45,1
6,test,credit_bureau_b,2,1,10,6,6,1
7,test,debitcard,1,1,10,6,6,1
8,test,deposit,1,1,10,5,5,1
9,test,other,1,1,10,7,7,1



体积最大的 10 个文件：


,file_name,split,table_group,depth,shard,size_mb,row_count,column_count
41,train_credit_bureau_a_1_1.parquet,train,credit_bureau_a,1,1,174.80,6009192,79
42,train_credit_bureau_a_1_2.parquet,train,credit_bureau_a,1,2,117.78,3743810,79
62,train_static_0_0.parquet,train,static,0,0,108.58,1003757,168
36,train_applprev_1_0.parquet,train,applprev,1,0,102.00,3887684,41
63,train_static_0_1.parquet,train,static,0,1,71.23,522902,168
37,train_applprev_1_1.parquet,train,applprev,1,1,69.49,2638295,41
43,train_credit_bureau_a_1_3.parquet,train,credit_bureau_a,1,3,68.45,2079323,79
40,train_credit_bureau_a_1_0.parquet,train,credit_bureau_a,1,0,61.86,4108212,79
50,train_credit_bureau_a_2_5.parquet,train,credit_bureau_a,2,5,58.16,33053760,19
49,train_credit_bureau_a_2_4.parquet,train,credit_bureau_a,2,4,46.95,27025737,19


In [9]:
#Step 1.5：Schema一致性审计
import json
import hashlib
import pandas as pd


# 1. 单独生成“只考虑列名”的指纹
def make_column_name_fingerprint(schema_json):
    schema_dict = json.loads(schema_json)
    column_names = sorted(schema_dict.keys())

    text = json.dumps(
        column_names,
        ensure_ascii=False
    )

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()[:16]


inventory["column_name_fingerprint"] = (
    inventory["schema_json"]
    .apply(make_column_name_fingerprint)
)


print("同一表结构内部的列名版本数：")

column_name_summary = (
    inventory.groupby(
        ["split", "table_group", "depth"],
        dropna=False
    )
    .agg(
        physical_file_count=("file_name", "count"),
        column_name_version_count=(
            "column_name_fingerprint",
            "nunique"
        ),
        full_schema_version_count=(
            "schema_fingerprint",
            "nunique"
        )
    )
    .reset_index()
)

display(column_name_summary)


# 2. 将每个文件的 schema 展开成长表
schema_records = []

for row in inventory.itertuples():
    schema_dict = json.loads(row.schema_json)

    for column_name, arrow_type in schema_dict.items():
        schema_records.append({
            "file_name": row.file_name,
            "split": row.split,
            "table_family_id": row.table_family_id,
            "column_name": column_name,
            "arrow_type": arrow_type
        })

schema_long = pd.DataFrame(schema_records)


# 3. 建立“表结构—数据集—字段”的类型集合
type_lookup = {}

for keys, group in schema_long.groupby(
    ["table_family_id", "split", "column_name"]
):
    type_lookup[keys] = set(group["arrow_type"])


audit_records = []
hard_type_details = []
test_null_only_details = []

for family_id in sorted(
    inventory["table_family_id"].unique()
):
    family_data = schema_long[
        schema_long["table_family_id"] == family_id
    ]

    train_columns = set(
        family_data.loc[
            family_data["split"] == "train",
            "column_name"
        ]
    )

    test_columns = set(
        family_data.loc[
            family_data["split"] == "test",
            "column_name"
        ]
    )

    train_only = train_columns - test_columns
    test_only = test_columns - train_columns

    # base表中target只存在于训练集，这是预期差异
    expected_train_only = (
        {"target"}
        if family_id == "base__depth_base"
        else set()
    )

    unexpected_train_only = train_only - expected_train_only
    unexpected_test_only = test_only

    hard_mismatch_count = 0
    test_null_only_count = 0

    for column_name in sorted(
        train_columns & test_columns
    ):
        train_types = type_lookup.get(
            (family_id, "train", column_name),
            set()
        )

        test_types = type_lookup.get(
            (family_id, "test", column_name),
            set()
        )

        # 排除纯null类型后再比较
        train_non_null_types = train_types - {"null"}
        test_non_null_types = test_types - {"null"}

        if (
            train_non_null_types
            and test_non_null_types
            and train_non_null_types != test_non_null_types
        ):
            hard_mismatch_count += 1

            hard_type_details.append({
                "table_family_id": family_id,
                "column_name": column_name,
                "train_types": sorted(train_types),
                "test_types": sorted(test_types)
            })

        elif (
            train_non_null_types
            and not test_non_null_types
            and "null" in test_types
        ):
            test_null_only_count += 1

            test_null_only_details.append({
                "table_family_id": family_id,
                "column_name": column_name,
                "train_types": sorted(train_types),
                "test_types": sorted(test_types)
            })

    audit_records.append({
        "table_family_id": family_id,
        "train_column_count": len(train_columns),
        "test_column_count": len(test_columns),
        "expected_train_only": "|".join(
            sorted(expected_train_only & train_only)
        ),
        "unexpected_train_only": "|".join(
            sorted(unexpected_train_only)
        ),
        "unexpected_test_only": "|".join(
            sorted(unexpected_test_only)
        ),
        "hard_type_mismatch_count": hard_mismatch_count,
        "test_null_only_count": test_null_only_count
    })


schema_cross_split_audit = pd.DataFrame(audit_records)

print("\nTrain/Test Schema 对比：")
display(schema_cross_split_audit)


print(
    "\n真正的非空类型冲突数量：",
    len(hard_type_details)
)

if hard_type_details:
    display(pd.DataFrame(hard_type_details))


print(
    "\n测试集纯 null 类型字段数量：",
    len(test_null_only_details)
)

if test_null_only_details:
    display(
        pd.DataFrame(test_null_only_details).head(30)
    )

同一表结构内部的列名版本数：


,split,table_group,depth,physical_file_count,column_name_version_count,full_schema_version_count
0,test,applprev,1,3,1,2
1,test,applprev,2,1,1,1
2,test,base,<NA>,1,1,1
3,test,credit_bureau_a,1,5,1,4
4,test,credit_bureau_a,2,12,1,1
5,test,credit_bureau_b,1,1,1,1
6,test,credit_bureau_b,2,1,1,1
7,test,debitcard,1,1,1,1
8,test,deposit,1,1,1,1
9,test,other,1,1,1,1



Train/Test Schema 对比：


,table_family_id,train_column_count,test_column_count,expected_train_only,unexpected_train_only,unexpected_test_only,hard_type_mismatch_count,test_null_only_count
0,applprev__depth_1,41,41,,,,0,0
1,applprev__depth_2,6,6,,,,0,1
2,base__depth_base,5,4,target,,,0,0
3,credit_bureau_a__depth_1,79,79,,,,1,0
4,credit_bureau_a__depth_2,19,19,,,,0,0
5,credit_bureau_b__depth_1,45,45,,,,0,1
6,credit_bureau_b__depth_2,6,6,,,,0,0
7,debitcard__depth_1,6,6,,,,0,0
8,deposit__depth_1,5,5,,,,0,0
9,other__depth_1,7,7,,,,0,0



真正的非空类型冲突数量： 5


,table_family_id,column_name,train_types,test_types
0,credit_bureau_a__depth_1,num_group1,[int64],"[double, int64]"
1,static__depth_0,validfrom_1069D,[string],[double]
2,static_cb__depth_0,assignmentdate_238D,[string],[double]
3,static_cb__depth_0,birthdate_574D,[string],[double]
4,static_cb__depth_0,responsedate_1012D,[string],[double]



测试集纯 null 类型字段数量： 17


,table_family_id,column_name,train_types,test_types
0,applprev__depth_2,credacc_cards_status_52L,[string],[null]
1,credit_bureau_b__depth_1,periodicityofpmts_997L,[string],[null]
2,person__depth_1,empl_employedtotal_800L,[string],[null]
3,person__depth_1,empl_industry_691L,[string],[null]
4,person__depth_2,empls_employedfrom_796D,[string],[null]
5,static__depth_0,equalityempfrom_62L,[bool],[null]
6,static__depth_0,lastrepayingdate_696D,[string],[null]
7,static_cb__depth_0,assignmentdate_4527235D,[string],[null]
8,static_cb__depth_0,dateofbirth_342D,[string],[null]
9,static_cb__depth_0,requesttype_4525192L,[string],[null]


In [10]:
#Step 1.6：确认冲突字段是否确实为空
import pandas as pd
import pyarrow.parquet as pq


issue_pairs = pd.DataFrame(
    hard_type_details + test_null_only_details
)[["table_family_id", "column_name"]].drop_duplicates()


verification_records = []

test_inventory = inventory[
    inventory["split"] == "test"
].copy()


for row in test_inventory.itertuples():
    relevant_columns = (
        issue_pairs.loc[
            issue_pairs["table_family_id"]
            == row.table_family_id,
            "column_name"
        ]
        .tolist()
    )

    if not relevant_columns:
        continue

    file_path = DATA_ROOT / row.relative_path
    available_columns = set(
        pq.ParquetFile(file_path).schema_arrow.names
    )

    columns_to_read = [
        column
        for column in relevant_columns
        if column in available_columns
    ]

    if not columns_to_read:
        continue

    small_df = pd.read_parquet(
        file_path,
        columns=columns_to_read
    )

    schema_dict = json.loads(row.schema_json)

    for column in columns_to_read:
        non_null_values = (
            small_df[column]
            .dropna()
            .drop_duplicates()
        )

        sample_values = (
            non_null_values
            .astype(str)
            .head(5)
            .tolist()
        )

        verification_records.append({
            "file_name": row.file_name,
            "table_family_id": row.table_family_id,
            "column_name": column,
            "stored_arrow_type": schema_dict[column],
            "row_count": len(small_df),
            "non_null_count": int(
                small_df[column].notna().sum()
            ),
            "sample_values": " | ".join(sample_values)
        })


type_issue_verification = pd.DataFrame(
    verification_records
)


print("冲突字段逐文件检查：")

display(
    type_issue_verification.sort_values(
        [
            "table_family_id",
            "column_name",
            "file_name"
        ]
    )
)


print("\n按字段汇总：")

type_issue_summary = (
    type_issue_verification.groupby(
        ["table_family_id", "column_name"],
        dropna=False
    )
    .agg(
        files_checked=("file_name", "count"),
        total_rows=("row_count", "sum"),
        total_non_null=("non_null_count", "sum"),
        observed_arrow_types=(
            "stored_arrow_type",
            lambda x: " | ".join(
                sorted(set(x))
            )
        )
    )
    .reset_index()
)

display(type_issue_summary)

冲突字段逐文件检查：


,file_name,table_family_id,column_name,stored_arrow_type,row_count,non_null_count,sample_values
0,test_applprev_2.parquet,applprev__depth_2,credacc_cards_status_52L,null,10,0,
1,test_credit_bureau_a_1_0.parquet,credit_bureau_a__depth_1,num_group1,int64,10,10,13 | 0 | 1 | 14 | 10
2,test_credit_bureau_a_1_1.parquet,credit_bureau_a__depth_1,num_group1,int64,10,10,11 | 1 | 0 | 2 | 12
3,test_credit_bureau_a_1_2.parquet,credit_bureau_a__depth_1,num_group1,int64,10,10,3 | 4 | 6 | 5 | 2
4,test_credit_bureau_a_1_3.parquet,credit_bureau_a__depth_1,num_group1,double,10,10,10.0 | 9.0 | 1.0 | 2.0 | 0.0
5,test_credit_bureau_a_1_4.parquet,credit_bureau_a__depth_1,num_group1,int64,10,10,4 | 2 | 0 | 3 | 1
6,test_credit_bureau_b_1.parquet,credit_bureau_b__depth_1,periodicityofpmts_997L,null,10,0,
7,test_person_1.parquet,person__depth_1,empl_employedtotal_800L,null,10,0,
8,test_person_1.parquet,person__depth_1,empl_industry_691L,null,10,0,
9,test_person_2.parquet,person__depth_2,empls_employedfrom_796D,null,10,0,



按字段汇总：


,table_family_id,column_name,files_checked,total_rows,total_non_null,observed_arrow_types
0,applprev__depth_2,credacc_cards_status_52L,1,10,0,null
1,credit_bureau_a__depth_1,num_group1,5,50,50,double | int64
2,credit_bureau_b__depth_1,periodicityofpmts_997L,1,10,0,null
3,person__depth_1,empl_employedtotal_800L,1,10,0,null
4,person__depth_1,empl_industry_691L,1,10,0,null
5,person__depth_2,empls_employedfrom_796D,1,10,0,null
6,static__depth_0,equalityempfrom_62L,3,30,0,null
7,static__depth_0,lastrepayingdate_696D,3,30,0,null
8,static__depth_0,validfrom_1069D,3,30,0,double
9,static_cb__depth_0,assignmentdate_238D,1,10,0,double


In [11]:
#Step 1.7：登记候选键和时间字段
import json
import pandas as pd


def get_candidate_key(row):
    """
    这里只登记候选键，不验证唯一性。
    """
    if row["table_role"] == "base":
        return "case_id"

    depth = row["depth"]

    if pd.isna(depth):
        return ""

    if int(depth) == 0:
        return "case_id"

    if int(depth) == 1:
        return "case_id + num_group1"

    if int(depth) == 2:
        return "case_id + num_group1 + num_group2"

    return ""


def get_time_fields(row):
    """
    识别日期字段和base表中的时间索引。
    """
    schema_dict = json.loads(row["schema_json"])
    column_names = schema_dict.keys()

    time_fields = [
        column
        for column in column_names
        if (
            column.endswith("D")
            or column in {
                "date_decision",
                "WEEK_NUM",
                "MONTH"
            }
        )
    ]

    return " | ".join(sorted(time_fields))


def candidate_key_is_complete(row):
    """
    检查候选键字段是否真实存在于schema中。
    """
    schema_dict = json.loads(row["schema_json"])
    column_names = set(schema_dict.keys())

    if not row["candidate_key"]:
        return False

    required_columns = row["candidate_key"].split(" + ")

    return all(
        column in column_names
        for column in required_columns
    )


inventory["candidate_key"] = inventory.apply(
    get_candidate_key,
    axis=1
)

inventory["time_fields"] = inventory.apply(
    get_time_fields,
    axis=1
)

inventory["candidate_key_complete"] = inventory.apply(
    candidate_key_is_complete,
    axis=1
)


print(
    "候选键字段不完整的文件数：",
    int((~inventory["candidate_key_complete"]).sum())
)


def combine_time_fields(series):
    combined = set()

    for value in series:
        if value:
            combined.update(value.split(" | "))

    return " | ".join(sorted(combined))


table_key_time_summary = (
    inventory.groupby(
        ["table_group", "depth", "table_role"],
        dropna=False
    )
    .agg(
        candidate_key=(
            "candidate_key",
            "first"
        ),
        candidate_key_complete=(
            "candidate_key_complete",
            "all"
        ),
        time_fields=(
            "time_fields",
            combine_time_fields
        )
    )
    .reset_index()
)

display(table_key_time_summary)

候选键字段不完整的文件数： 0


,table_group,depth,table_role,candidate_key,candidate_key_complete,time_fields
0,applprev,1,feature_table,case_id + num_group1,True,approvaldate_319D | creationdate_885D | dateac...
1,applprev,2,feature_table,case_id + num_group1 + num_group2,True,
2,base,<NA>,base,case_id,True,MONTH | WEEK_NUM | date_decision
3,credit_bureau_a,1,feature_table,case_id + num_group1,True,dateofcredend_289D | dateofcredend_353D | date...
4,credit_bureau_a,2,feature_table,case_id + num_group1 + num_group2,True,
5,credit_bureau_b,1,feature_table,case_id + num_group1,True,contractdate_551D | contractmaturitydate_151D ...
6,credit_bureau_b,2,feature_table,case_id + num_group1 + num_group2,True,pmts_date_1107D
7,debitcard,1,feature_table,case_id + num_group1,True,openingdate_857D
8,deposit,1,feature_table,case_id + num_group1,True,contractenddate_991D | openingdate_313D
9,other,1,feature_table,case_id + num_group1,True,


In [12]:
#Step 1.8：生成两个规定产物
from datetime import datetime
import pandas as pd


# ==================================================
# 一、生成 metadata/file_inventory.csv
# 一行代表一个Parquet物理文件，共68行
# ==================================================

inventory_columns = [
    "file_name",
    "relative_path",
    "size_bytes",
    "size_mb",
    "split",
    "table_group",
    "depth",
    "shard",
    "table_role",
    "is_sharded",
    "table_family_id",
    "row_count",
    "row_group_count",
    "column_count",
    "has_case_id",
    "has_target",
    "candidate_key",
    "candidate_key_complete",
    "time_fields",
    "column_name_fingerprint",
    "schema_fingerprint",
    "schema_json"
]

file_inventory = (
    inventory[inventory_columns]
    .sort_values(
        ["split", "table_group", "depth", "shard"],
        na_position="first"
    )
    .reset_index(drop=True)
)

file_inventory_path = (
    METADATA_DIR / "file_inventory.csv"
)

file_inventory.to_csv(
    file_inventory_path,
    index=False,
    encoding="utf-8-sig"
)


# ==================================================
# 二、汇总成17种逻辑表目录
# ==================================================

def combine_time_fields(series):
    result = set()

    for value in series.dropna():
        if value:
            result.update(value.split(" | "))

    return sorted(result)


def count_label(dataframe):
    values = sorted(
        dataframe["column_count"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    if not values:
        return ""

    if len(values) == 1:
        return str(values[0])

    return " / ".join(map(str, values))


catalog_records = []

for table_family_id, group in inventory.groupby(
    "table_family_id",
    sort=True
):
    train_group = group[group["split"] == "train"]
    test_group = group[group["split"] == "test"]

    depth_values = group["depth"].dropna()

    if depth_values.empty:
        depth_label = "base"
    else:
        depth_label = str(int(depth_values.iloc[0]))

    combined_time_fields = combine_time_fields(
        group["time_fields"]
    )

    catalog_records.append({
        "table_family_id": table_family_id,
        "table_group": group["table_group"].iloc[0],
        "depth": depth_label,
        "train_files": len(train_group),
        "test_files": len(test_group),
        "train_rows": int(
            train_group["row_count"].sum()
        ),
        "test_rows": int(
            test_group["row_count"].sum()
        ),
        "train_columns": count_label(train_group),
        "test_columns": count_label(test_group),
        "candidate_key": group["candidate_key"].iloc[0],
        "candidate_key_complete": bool(
            group["candidate_key_complete"].all()
        ),
        "train_schema_versions": int(
            train_group["schema_fingerprint"].nunique()
        ),
        "test_schema_versions": int(
            test_group["schema_fingerprint"].nunique()
        ),
        "time_field_count": len(combined_time_fields),
        "time_fields": combined_time_fields
    })

table_catalog = pd.DataFrame(catalog_records)


# ==================================================
# 三、把目录表转换成Markdown
# 不依赖额外安装tabulate
# ==================================================

catalog_display = table_catalog[
    [
        "table_group",
        "depth",
        "train_files",
        "test_files",
        "train_rows",
        "test_rows",
        "train_columns",
        "test_columns",
        "candidate_key",
        "candidate_key_complete",
        "train_schema_versions",
        "test_schema_versions",
        "time_field_count"
    ]
].rename(columns={
    "table_group": "表组",
    "depth": "Depth",
    "train_files": "Train文件数",
    "test_files": "Test文件数",
    "train_rows": "Train物理行数",
    "test_rows": "Test物理行数",
    "train_columns": "Train字段数",
    "test_columns": "Test字段数",
    "candidate_key": "候选键",
    "candidate_key_complete": "候选键字段完整",
    "train_schema_versions": "Train Schema版本数",
    "test_schema_versions": "Test Schema版本数",
    "time_field_count": "时间字段数"
})


def escape_markdown(value):
    return (
        str(value)
        .replace("|", "\\|")
        .replace("\n", " ")
    )


def dataframe_to_markdown(dataframe):
    headers = [
        escape_markdown(column)
        for column in dataframe.columns
    ]

    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(
            ["---"] * len(headers)
        ) + " |"
    ]

    for row in dataframe.itertuples(index=False):
        values = [
            escape_markdown(value)
            for value in row
        ]

        lines.append(
            "| " + " | ".join(values) + " |"
        )

    return "\n".join(lines)


# ==================================================
# 四、编写 metadata/table_catalog.md
# ==================================================

train_base_rows = int(
    inventory.loc[
        inventory["file_name"] == "train_base.parquet",
        "row_count"
    ].iloc[0]
)

catalog_lines = [
    "# Home Credit 2024 文件级表目录",
    "",
    f"- 生成时间：{datetime.now().isoformat(timespec='seconds')}",
    f"- 数据根目录：`{DATA_ROOT}`",
    f"- Parquet物理文件数：{len(file_inventory)}",
    f"- 逻辑表结构数：{len(table_catalog)}",
    (
        "- Parquet总大小："
        f"{inventory['size_bytes'].sum() / 1024**3:.2f} GB"
    ),
    f"- Train base行数：{train_base_rows:,}",
    "- 原始数据处理原则：只读",
    "",
    "## 1. 审计结论",
    "",
    (
        "- 68个Parquet物理文件对应17种逻辑表结构，"
        "分片不能视为独立业务表。"
    ),
    "- Train包含32个文件，Test包含36个文件。",
    (
        "- 所有同组分片的字段名一致；"
        "Train各表组内部schema一致。"
    ),
    (
        "- Test为10个case的无标签样例，"
        "部分全空字段存在物理类型变化，"
        "后续读取时需要显式统一类型。"
    ),
    (
        "- 候选键仅根据base/depth结构登记，"
        "本步骤未验证唯一性。"
    ),
    (
        "- 所有文件均包含其所属层级需要的"
        "候选键字段。"
    ),
    "",
    "## 2. 逻辑表目录",
    "",
    dataframe_to_markdown(catalog_display),
    "",
    "## 3. 时间字段目录",
    ""
]

for row in table_catalog.itertuples(index=False):
    if row.time_fields:
        fields_text = "、".join(
            f"`{field}`"
            for field in row.time_fields
        )
    else:
        fields_text = "未识别到时间字段"

    catalog_lines.append(
        f"- `{row.table_family_id}`：{fields_text}"
    )

catalog_lines.extend([
    "",
    "## 4. 字段定义文件",
    "",
    (
        f"- `feature_definitions.csv`包含"
        f"{len(feature_definitions)}条字段定义。"
    ),
    "- 字段列为`Variable`和`Description`。",
    "- 两列均无缺失值。",
    "",
    "## 5. 本步骤边界",
    "",
    "- 未检查`case_id`唯一性和重复客户；",
    "- 未检查时间范围和周度样本；",
    "- 未统计target-event rate；",
    "- 未设计Train、Validation和Final OOT；",
    "- 未拼表、清洗、构造特征或训练模型。",
    "",
    "以上内容留待Step 2及后续步骤。"
])

table_catalog_path = (
    METADATA_DIR / "table_catalog.md"
)

table_catalog_path.write_text(
    "\n".join(catalog_lines),
    encoding="utf-8"
)


# ==================================================
# 五、验收
# ==================================================

print("Step 1 输出文件：")
print("1.", file_inventory_path)
print("2.", table_catalog_path)

print("\n文件是否成功生成：")
print(
    "file_inventory.csv：",
    file_inventory_path.exists()
)
print(
    "table_catalog.md：",
    table_catalog_path.exists()
)

print("\n输出行数：")
print(
    "file_inventory.csv 数据行数：",
    len(file_inventory)
)
print(
    "table_catalog.md 逻辑表数：",
    len(table_catalog)
)

print("\n候选键字段不完整文件数：")
print(
    int(
        (~inventory["candidate_key_complete"])
        .sum()
    )
)

Step 1 输出文件：
1. <PROJECT_ROOT>\metadata\file_inventory.csv
2. <PROJECT_ROOT>\metadata\table_catalog.md

文件是否成功生成：
file_inventory.csv： True
table_catalog.md： True

输出行数：
file_inventory.csv 数据行数： 68
table_catalog.md 逻辑表数： 17

候选键字段不完整文件数：
0
